#The Transformation Logic

In [0]:
querry = """
select
    ROW_NUMBER() OVER(ORDER BY ci.customer_id) as customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.firstname,
    ci.lastname,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.create_date AS create_date
from workspace.silver.crm_customers ci
left join workspace.silver.erp_customer_location la
on ci.customer_number = la.customer_number
left join workspace.silver.erp_customers ca
on ci.customer_number = ca.customer_number
"""
df= spark.sql(querry)

In [0]:
df.limit(10).display()

#Writing Into Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

##Sanity Check For Gold Table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10